<a href="https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**Task Type:** Ranking and Scoring via Supervised Classification.

This is fundamentally a prioritization problem. While the underlying model will perform binary classification (predicting whether a page is a strong candidate for a refresh), the ultimate task is to output a ranked queue. We are scoring pages to sort them from highest opportunity/risk to lowest.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**The Target Proxy:**
In a perfect time-series dataset, the true target would be "future traffic decline." Because the starter dataset is a current-state snapshot, I will use a proxy label: `is_opportunity`.

For this baseline framing, a page is a positive target (`1`) if it has established visibility (e.g., `impressions_90d > 500`) but is showing weakness (`trend_direction == "down"` OR `ctr < 0.5`).


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**Success Metric:** Precision@K (specifically, Precision@50).

Standard accuracy or AUC-ROC is not the best fit for a human-in-the-loop workflow. If the content team only has the capacity to review 50 pages this week, we only care about the density of true positives in our top 50 predictions. Recall does not matter if the team cannot realistically edit all the flagging pages anyway.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# 1. Download the repo into Colab
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/Tamur-Naseem/FLY-RANK.git
    os.chdir("FLY-RANK")

# 2. Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 3. Define the unit of analysis (One row = One Content Item)
print(f"The unit of analysis is a single content item (page).")
print(f"The dataframe has {df.shape[0]:,} rows (pages) and {df.shape[1]} features.")

# 4. Sketch the target proxy column
# Creating a simple proxy: It has traffic but is trending down
df['is_opportunity_proxy'] = ((df['impressions_90d'] > 500) & (df['trend_direction'] == 'down')).astype(int)

# Display the grain of the data and the new proxy target
display_cols = ['content_id', 'impressions_90d', 'trend_direction', 'is_opportunity_proxy']
df[display_cols].head()


Cloning into 'FLY-RANK'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 129 (delta 42), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.85 MiB | 17.20 MiB/s, done.
Resolving deltas: 100% (42/42), done.
The unit of analysis is a single content item (page).
The dataframe has 30,000 rows (pages) and 44 features.


,content_id,impressions_90d,trend_direction,is_opportunity_proxy
0,content_304f48230142,3803,down,1
1,content_a1fb4e703a9e,15320,down,1
2,content_9aa793d4d895,12581,down,1
3,content_331d6c4de07b,11751,stable,0
4,content_d99b7a2d90ca,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**Why ML Beats a Fixed Rule:**
A fixed heuristic (like `IF age > 180 AND trend == down THEN refresh`) is brittle. It treats a page with 501 impressions exactly the same as a page with 50,000 impressions, and it ignores nuanced interactions between features like engagement rate, average position, and search volume.

A machine learning model can weight these continuous and categorical variables simultaneously, identifying subtle patterns—such as a page with moderate age but an abnormally fast-decaying CTR—that a hardcoded SQL rule would completely miss.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.